In [ ]:
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import sys, os
sys.path.append(os.path.abspath("../data_creation/plots_and_analysis"))
import config
# adjust paths in /data_creation/plots_and_analysis/config.py


In [ ]:
config.path_to_evaluation

In [ ]:
# get accuracy metrics
all_metrics_df = pd.read_parquet(f"{config.path_to_evaluation}/results.parquet")
all_metrics_df["div_step"] = all_metrics_df["div_step"].astype(str)
all_metrics_df.rename(columns={"div_step": "div", "pot target size": "p"}, inplace=True)
all_metrics_df


In [ ]:
# append target features
mean_target_features = pd.read_parquet(f"{config.path_to_evaluation}target_features_num_train_mean.parquet") 
all_metrics_df = all_metrics_df.merge(mean_target_features,on=["p", "div", "run"], how="outer")
all_metrics_df


In [ ]:
all_metrics_df.columns

In [ ]:
# load source features
mean_source_features = pd.read_parquet(f"{config.path_to_evaluation}/source_features_num_train_mean.parquet")
mean_source_features

In [ ]:
# merge with source features
# Merge the dataframes on 'div' and 'run' keys
merged_df = all_metrics_df.merge(
    mean_source_features, 
    on=['div', 'run'], 
    how='left', 
    suffixes=('', '_source')
)

# Fill missing values in rows where p='source'
mask = merged_df['p'] == 'source'

feature_columns = set(mean_source_features.columns) - set(["div", "run"])

#feature_columns = [col for col in all_metrics_df.columns if col not in ['RMSSE', 'MASE', 'sMAPE', 'p', 'similarity', 'div', 'run']]

for col in feature_columns:
    if f'{col}_source' in merged_df.columns:
        merged_df.loc[mask, col] = merged_df.loc[mask, col].fillna(merged_df.loc[mask, f'{col}_source'])

source_cols = [col for col in merged_df.columns if col.endswith('_source')]
merged_df = merged_df.drop(columns=source_cols)

all_metrics_df = merged_df
all_metrics_df.rename(columns={"agg_autocorrelation_max":"autocorr", "agg_linear_trend_slope":"trend", "standard_deviation":"sd"}, inplace=True)
all_metrics_df.to_parquet(f"{config.path_to_evaluation}/results_metrics_features_train_mean.parquet")
all_metrics_df

To avoid collinearity in the linear regression, calculate the Pearson correlation matrix.

In [ ]:
features = ['abs_energy', 'intermittency', 'mean', 'median', 'kurtosis', 
                     'skewness', 'sd', 'autocorr', 
                     'erraticness', 'trend']


div = "aggregated" #choose between: "0", "5", "10", "aggregated"

if div == "aggregated":
    complete_data = all_metrics_df[features].dropna() 
else:
    complete_data = all_metrics_df.loc[all_metrics_df["div"] == div][features].dropna()
    
correlation_matrix = complete_data.corr()

plt.rcParams.update({
    'font.size': 12,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'axes.linewidth': 2
})

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', 
            annot_kws={'weight': 'bold', 'size': 10}) 
plt.tight_layout()  
plt.savefig(f"{config.path_to_evaluation}/feature_correlations_div{div}_train_mean.pdf",  bbox_inches='tight')
plt.show()

 Plot single features against the metrics. Fit a linear line and calculate the slope and p-value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(
    4,
    2,
    figsize=(14, 24)
)

axes = axes.flatten()

df = all_metrics_df

unique_pot_target_sizes = df["p"].unique()
colors = plt.cm.tab20(
    np.linspace(0, 1, len(unique_pot_target_sizes))
)


def plot_linear_fit(ax, x, y, metric):

    if len(x) > 1:

        # Linear regression
        linear_coeffs = np.polyfit(x, y, deg=1)
        linear_eq = np.poly1d(linear_coeffs)

        x_fit = np.linspace(
            np.nanmin(x),
            np.nanmax(x),
            100
        )

        y_fit = linear_eq(x_fit)

        ax.plot(
            x_fit,
            y_fit,
            linestyle="--",
            alpha=0.9,
            linewidth=4,
            color="black",
        )

        for i, size in enumerate(unique_pot_target_sizes):

            mask = df["p"] == size

            ax.scatter(
                x[mask],
                y[mask],
                c=[colors[i]],
                s=150,
                edgecolors="w",
                alpha=0.9,
                linewidth=1.5,
            )

        slope, intercept, r_value, p_value, std_err = stats.linregress(
            x,
            y
        )

        ax.annotate(
            f"Slope: {slope:.3f}",
            xy=(0.5, 0.98),
            xycoords="axes fraction",
            fontsize=14,
            ha="center",
            va="top",
            bbox=dict(
                boxstyle="round,pad=0.3",
                edgecolor="black",
                facecolor="lightgray",
            ),
        )


# --------------------------------------------------
# Plot definitions
# --------------------------------------------------

plot_specs = [
    ("intermittency", "sMAPE"),
    ("erraticness", "RMSSE"),
    ("erraticness", "MASE"),
    ("skewness", "MASE"),
    ("skewness", "RMSSE"),
    ("autocorr", "sMAPE"),
    ("kurtosis", "MASE"),
    ("kurtosis", "RMSSE"),
]

# --------------------------------------------------
# Create plots
# --------------------------------------------------

for ax, (feature, metric) in zip(axes, plot_specs):

    plot_linear_fit(
        ax,
        df[feature],
        df[metric],
        metric,
    )

    ax.set_xlabel(
        feature.replace("_", " ").title(),
        fontsize=18,
    )

    ax.set_ylabel(
        metric,
        fontsize=18,
    )

    ax.set_title(
        f"{feature.replace('_', ' ').title()} vs. {metric}",
        fontsize=18,
    )

    ax.grid(True, alpha=0.3)

    ax.tick_params(
        axis="both",
        labelsize=14,
    )

# --------------------------------------------------
# Legend
# --------------------------------------------------

legend_elements = []

for i, size in enumerate(unique_pot_target_sizes):

    legend_elements.append(
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=colors[i],
            markersize=10,
            markeredgecolor="white",
            markeredgewidth=0.5,
            label=str(size),
        )
    )

fig.legend(
    title="Potential Target Size",
    handles=legend_elements,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.05),
    ncol=len(unique_pot_target_sizes),
    fontsize=14,
    frameon=True,
    title_fontsize=16,
)

plt.tight_layout()

plt.subplots_adjust(
    bottom=0.12,
    hspace=0.30,
    wspace=0.25,
)

plt.savefig(
    f"{config.path_to_evaluation}/single_ols.pdf",
    bbox_inches="tight",
)

plt.show()